# Basic Read/Write

Define a model, build a graph, read it back, and write it out.

## Namespaces & Models

In [1]:
from rdflib import RDF, XSD, Graph, Literal, Namespace

from rdfantic import GraphModel, predicate

SCHEMA = Namespace("http://schema.org/")
EX = Namespace("http://example.org/")


class PersonView(GraphModel):
    rdf_type = SCHEMA["Person"]
    name: str = predicate(SCHEMA["name"])
    email: str | None = predicate(SCHEMA["email"])


class MovieView(GraphModel):
    rdf_type = SCHEMA["Movie"]
    name: str = predicate(SCHEMA["name"])
    director: PersonView = predicate(SCHEMA["director"])
    genres: set[str] = predicate(SCHEMA["genre"])
    year: int | None = predicate(SCHEMA["year"])

## Build a small graph

In [2]:
g = Graph()
g.bind("schema", SCHEMA)
g.bind("ex", EX)

# A person
g.add((EX["nolan"], RDF.type, SCHEMA["Person"]))
g.add((EX["nolan"], SCHEMA["name"], Literal("Christopher Nolan", datatype=XSD.string)))
g.add((EX["nolan"], SCHEMA["email"], Literal("nolan@example.org", datatype=XSD.string)))

# A movie
g.add((EX["inception"], RDF.type, SCHEMA["Movie"]))
g.add((EX["inception"], SCHEMA["name"], Literal("Inception", datatype=XSD.string)))
g.add((EX["inception"], SCHEMA["director"], EX["nolan"]))
g.add((EX["inception"], SCHEMA["genre"], Literal("Sci-Fi", datatype=XSD.string)))
g.add((EX["inception"], SCHEMA["genre"], Literal("Thriller", datatype=XSD.string)))
g.add((EX["inception"], SCHEMA["year"], Literal(2010, datatype=XSD.integer)))

<Graph identifier=N93cb48e2419a4ced9194c4ce5c3b5813 (<class 'rdflib.graph.Graph'>)>

## Read from graph

In [3]:
movie = MovieView.from_graph(g, EX["inception"])

print(f"Title:    {movie.name}")
print(f"Year:     {movie.year}")
print(f"Genres:   {sorted(movie.genres)}")
print(f"Director: {movie.director.name}")
print(f"Email:    {movie.director.email}")
print(f"Subject:  {movie.subject}")

Title:    Inception
Year:     2010
Genres:   ['Sci-Fi', 'Thriller']
Director: Christopher Nolan
Email:    nolan@example.org
Subject:  http://example.org/inception


## Write to a new graph

In [4]:
g2 = Graph()
movie.to_graph(g2, subject=EX["inception"])

print(f"Triples written: {len(g2)}")
print()
for s, p, o in sorted(g2):
    print(f"  {s} {p} {o}")

Triples written: 9

  http://example.org/inception http://schema.org/director http://example.org/nolan
  http://example.org/inception http://schema.org/genre Sci-Fi
  http://example.org/inception http://schema.org/genre Thriller
  http://example.org/inception http://schema.org/name Inception
  http://example.org/inception http://schema.org/year 2010
  http://example.org/inception http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://schema.org/Movie
  http://example.org/nolan http://schema.org/email nolan@example.org
  http://example.org/nolan http://schema.org/name Christopher Nolan
  http://example.org/nolan http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://schema.org/Person


## Round-trip check

In [5]:
movie2 = MovieView.from_graph(g2, EX["inception"])
assert movie2.name == movie.name
assert movie2.year == movie.year
assert movie2.genres == movie.genres
assert movie2.director.name == movie.director.name
print("Round-trip OK!")

Round-trip OK!
